In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
import requests
from minsearch import Index

In [7]:
def build_index(documents):
    index = Index(
    text_fields=["content"],
    keyword_fields=["filename"])
    index.fit(documents)
    return index

In [35]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
index = build_index(chunks)

In [37]:
def search(query):

    return index.search(
        query,
        num_results=10
    )

In [38]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the lessons."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [12]:
import json
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import os
openai_client = OpenAI(
    api_key="yourkey",
    base_url="https://openrouter.ai/api/v1",
)

In [32]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    input_tokens = 0
    output_tokens = 0
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model="openrouter/free",
            input=messages,
           tools=[search_tool]
        )
        usage = response.usage
        input_tokens = input_tokens + usage.input_tokens;
        output_tokens = output_tokens + usage.output_tokens;
        #response = openai_client.chat.completions.create(
        #model="llama-3.3-70b-versatile",
        #input = messages,
        #tools = [search_tool],
        #)
        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                print("input tokens = " + str(input_tokens))
                print("output tokens = " + str(output_tokens))
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    return last_answer

In [33]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using lessons, don't do it yourself. Only use the 
facts from the lessons database.

At the end, ask if there are other areas that the user wants to explore.
"""

In [16]:
question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I’m sorry, but that question isn’t related to the course or its logistics. As a course teaching assistant, I’m only able to answer questions that fall within the scope of the course material or administrative details.

If you have any questions about the course content, enrollment, schedule, assignments, or anything else directly tied to the course, feel free to ask!


In [20]:
question = "How many lesson pages ?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"lesson pages course number of lesson pages"}
iteration #2...
function_call: search {"query":"pages"}
iteration #3...
function_call: search {"query":"lessons/"}
iteration #4...
ASSISTANT:

Let me search for the specific count of lesson pages:


function_call: search {"query": "how many lessons"}
iteration #5...
function_call: search {"query": "total lessons FAQ entries"}
iteration #6...
function_call: search {"query":"total documents FAQ"}
iteration #7...
function_call: search {"query":"COURSE LecPageID, COUNT(1) AS total_pages FROM COURSE WHERE LecPageType = 'lesson' GROUP BY LecPageID ORDER BY total_pages DESC"}
iteration #8...
ASSISTANT:



function_call: search {"query": "course lessons directory structure"}
iteration #9...
ASSISTANT:



function_call: search {"query": "module count modules lessons"}
iteration #10...
ASSISTANT:
Based on my search through the course materials, I can see the LLM Zoomcamp course is organized into multiple

In [39]:
question = "How does the agentic loop keep calling the model until it stops?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query": "agentic loop model calling stopping condition"}
iteration #2...
ASSISTANT:
input tokens = 6381
output tokens = 1208


The agenticloop continues calling the model iteratively until the model **explicitly stops making function calls** in its response. Here's how it works:

1. **Core Mechanism**:  
   The loop runs indefinitely (`while True`) and checks each model response for function calls. If a response contains function calls (e.g., `search`), the loop:  
   - Executes those calls.  
   - Appends the results to the conversation history.  
   - Continues to the next iteration.  

2. **Exit Condition**:  
   The loop stops when the model returns a response **without any function calls**. This signals the agent to finalize its answer. A flag (`has_function_calls`) tracks whether any calls were made in the current iteration. If no calls are made, the loop breaks.

3. **Code Example**:  
   From the search results:  
   ```python
   has_func

In [22]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [24]:
len(chunks)

295

In [25]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

In [26]:
question = "How does the agentic loop work, and how is it different from plain RAG?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"agentic loop"}
iteration #2...
ASSISTANT:
**Agentic loop – how it works**

The agentic loop is a `while True` cycle that puts the large language model (LLM) in charge of deciding what to do next.  
Its three core ingredients are:

1. **Instructions** (a `developer` or system message) that tell the model its role and how to behave.  
2. **Tools** – functions the model can call (e.g., a `search` function that looks up information in a knowledge base).  
3. **Memory** – the growing message history that records every user prompt, model output, and tool result.

In each iteration:

1. The model receives the current message history (instructions + user question + all prior tool results).  
2. It produces a response that may contain:
   * **tool calls** (e.g., “call `search` with this query”) or  
   * **final answer** (a plain text message).
3. The loop executes any requested tools, appends their outputs to the history, and then sends the updat